In [1]:
%matplotlib notebook
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

# Set seaborn style
sns.set() 

# Ejemplo de cómo propagar incertezas mediante Monte Carlo


Se desea calcular la eficiencia de un detector de radiación gamma para una determinada energía mediante la medición de una fuente patrón:

\begin{equation}
    \epsilon = \frac{C / t}{A \Gamma}
\end{equation}

donde:

- $\epsilon$: eficiencia del detector para la energía del gamma analizado
 
- C: cuentas netas bajo el pico de la energía del gamma analizada
 
- t: tiempo que duró la medición
 
- A: actividad de la fuente patrón
 
- $\Gamma$: Probabilidad de emisión del gamma analizado (branching ratio)

In [41]:
# Asumo estadística de Poisson
cuentas_mu = 10000

# Tiempo [s]
tiempo_mu = 600
tiempo_sig = 5

# Actividad de la fuente [Bq]
actividad_mu = 1e4
actividad_sig = actividad_mu * 0.03  # 3% de incerteza

# Probabilidad de emisión
gamma_mu = 0.80
gamma_sig = 0.02

num_samples = 10000

# Se simulan las variables con distribución normal

cuentas = np.random.poisson(cuentas_mu, num_samples)
tiempo = np.random.normal(tiempo_mu, tiempo_sig, num_samples)
actividad = np.random.normal(actividad_mu, actividad_sig, num_samples)
gamma = np.random.normal(gamma_mu, gamma_sig, num_samples)

# Se realiza la suma
eficiencia = cuentas / tiempo / actividad / gamma

In [42]:
eficiencia_mu = np.mean(eficiencia)
eficiencia_sig = np.std(eficiencia)
print(fr"El valor de la eficiencia es: {eficiencia_mu:.3e} $\pm$ {eficiencia_sig:.3e}")

El valor de la eficiencia es: 2.086e-03 $\pm$ 8.601e-05


# Utilizando el packete *uncertainties*

Es un paquete de python que utiliza la fórmula de propagación de errores a primer orden.

In [43]:
from uncertainties import ufloat

cuentas = ufloat(cuentas_mu, cuentas_sig, tag='cuentas')
tiempo = ufloat(tiempo_mu, tiempo_sig, tag='tiempo')
actividad = ufloat(actividad_mu, actividad_sig, tag='actividad')
gamma = ufloat(gamma_mu, gamma_sig, tag='gamma')

eficiencia = cuentas / tiempo / actividad / gamma
print(fr"El valor de la eficiencia es: {eficiencia:.3e}")

El valor de la eficiencia es: (2.083+/-0.086)e-03


In [44]:
# Se puede hacer un desgloce de la contribución de las incertezas de cada magnitud
for var, error_prop in eficiencia.error_components().items():
    porcentaje = (error_prop**2 / eficiencia.std_dev**2) * 100
    print(f"Origen: {var.tag}")
    print(f" - Contribución al desvío: {error_prop:.1e}")
    print(f" - Contribución porcentual: {porcentaje:.1f}%")


Origen: gamma
 - Contribución al desvío: 5.2e-05
 - Contribución porcentual: 36.9%
Origen: actividad
 - Contribución al desvío: 6.3e-05
 - Contribución porcentual: 53.1%
Origen: tiempo
 - Contribución al desvío: 1.7e-05
 - Contribución porcentual: 4.1%
Origen: cuentas
 - Contribución al desvío: 2.1e-05
 - Contribución porcentual: 5.9%
